# Import Packages

In [1]:
import math
from collections import Counter
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer

import torch
import torch.nn as nn

from transformers import set_seed

set_seed(42)

# Load dataset and Tokenizer

Loading the dataset, and setup a tokenizer, using <i>bert-base-multilingual-cased</i>. 

In [2]:
dataset = load_dataset("coastalcph/tydi_xor_rc")
df_train = dataset["train"].to_pandas()
df_validation = dataset["validation"].to_pandas()
df_train_ko = df_train[df_train['lang'] == 'ko']
df_val_ko = df_validation[df_validation['lang'] == 'ko']

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-multilingual-cased", use_fast=True
)


# Tokenize questions and create vocabulary

Tokenizing questions and creating a vocabulary and only keeping tokens which appears four times or more in the dictionary, additionally adding the \<s\>, \<\/s\>, \<unk\> tokens to the vocabulary. 

Questions containing tokens which does not appear in the pruned vocabulary, said tokens will be switched out with the \<unk\> token. 


In [3]:
def tokenizeQuestion(df):
    return [tokenizer.tokenize(question) for question in df['question']]

t_questions = tokenizeQuestion(df_train_ko)
vocabulary = Counter(token for question in t_questions for token in question)
vocab = {token for token, count in vocabulary.items() if count >= 4}
vocab.update(['<s>', '</s>', '<unk>'])

def applyUnk(tokenizedQuestions, vocab):
    return [[token if token in vocab else '<unk>' for token in q]
            for q in tokenizedQuestions]

t_questions_filtered = applyUnk(t_questions, vocab)
val_unk = applyUnk(tokenizeQuestion(df_val_ko), vocab)

print('Training questions:', len(t_questions_filtered))
print('Validation questions:', len(val_unk))
print('Vocabulary size (including special tokens):', len(vocab))

Training questions: 2422
Validation questions: 356
Vocabulary size (including special tokens): 920


# Unigram Baseline Model implementation

In [4]:
class Unigram():
	def __init__ (self):
		self.count = Counter(token for question in t_questions_filtered for token in question + ['</s>'])
		self.total = sum(self.count.values())

	def probability(self, token):
		return self.count[token] / self.total

	def perplexity(self, questions):
		log_probability = 0
		token_count = 0
		for question in questions:
			for token in question + ["</s>"]:
				log_probability += math.log(self.probability(token))
				token_count += 1
		return math.exp(-log_probability / token_count)

unigram = Unigram()


unigram_ppl = unigram.perplexity(val_unk)
print(f'Unigram validation perplexity: {unigram_ppl:.2f}')


Unigram validation perplexity: 185.07


# Smoothed Bigram

Implementing a smoothed bigram model, using <i>add-1</i> smoothing

In [5]:
class Bigram():
	def __init__(self):
		self.bigram_count = Counter()
		self.context_count = Counter()

	def train(self, sentences):
		for sentence in sentences:
			tokens = ['<s>'] + sentence + ['</s>']
			for i in range(1, len(tokens)):
				context = tokens[i-1]
				token = tokens[i]
				self.bigram_count[(context, token)] += 1
				self.context_count[context] += 1

	def probability(self, token, previous):
		return (self.bigram_count[(previous, token)] + 1) / (self.context_count[previous] + len(vocab))

	def perplexity(self, questions):
		log_probability = 0
		token_count = 0
		for question in questions:
			tokens = ['<s>'] + question + ['</s>']
			for previous, token in zip(tokens, tokens[1:]):
				log_probability += math.log(self.probability(token, previous))
				token_count += 1
		return math.exp(-log_probability / token_count)

bigram = Bigram()
bigram.train(t_questions_filtered)
bigram_ppl = bigram.perplexity(val_unk)

print(f'Smoothed bigram validation perplexity: {bigram_ppl:.2f}')


Smoothed bigram validation perplexity: 52.21


# Smoothed Trigram

Implementation of Trigram language model using <i>add-k smoothing</i>. 

In [6]:
class Trigram():
	def __init__(self):
		self.trigram_count = Counter()
		self.bigram_count = Counter()



	def train(self, sentences):
		for sentence in sentences:
			tokens = ["<s>", "<s>"] + sentence + ["</s>"]
			for i in range(2, len(tokens)):
				context = (tokens[i-2], tokens[i-1])
				token = tokens[i]
				self.trigram_count[(*context, token)] += 1
				self.bigram_count[context] += 1
    
	def probability(self, token, t1, t2, k):
		context = (t1, t2)
		numerator = self.trigram_count[(*context, token)] + k
		denominator = self.bigram_count[context] + (k * len(vocab))
		if denominator == 0:
			return 0
		return numerator / denominator
    
	def perplexity(self, sentences, k):
		log_prob = 0
		total_tokens = 0
		for sentence in sentences:
			tokens = ["<s>","<s>"] + sentence + ["</s>"]
			for i in range(2, len(tokens)):
				t1 = tokens[i-2]
				t2 = tokens[i-1]
				token = tokens[i]
				p = self.probability(token, t1, t2, k)
				if p == 0:
					return float("inf")
				log_prob += math.log(p)
				total_tokens += 1
		
		if total_tokens == 0:
			return float("inf")
		return math.exp(-log_prob / total_tokens)



trigram = Trigram()
trigram.train(t_questions_filtered)

trigram_results = {}
for k in [1, 0.1, 0.01, 0.001]:
	trigram_ppl = trigram.perplexity(val_unk, k=k)
	trigram_results[k] = trigram_ppl
	print(f"Trigram validation perplexity at k {k}: {trigram_ppl:.2f}")


Trigram validation perplexity at k 1: 103.84
Trigram validation perplexity at k 0.1: 43.01
Trigram validation perplexity at k 0.01: 25.76
Trigram validation perplexity at k 0.001: 24.73


# Feedforward Neural Trigram

Implementation of the feedforward neural trigram. implemented using <i>torch</i>

In [7]:
class FF_Trigram(nn.Module):
	def __init__(self, vocab, context_size=2, embed_dim=16, hidden_dim=32):
		super().__init__()
		self.vocab_size = len(vocab)
		self.context_size = context_size
		self.embedding = nn.Embedding(self.vocab_size, embed_dim)
		self.hidden = nn.Linear(context_size * embed_dim, hidden_dim)
		self.output = nn.Linear(hidden_dim, self.vocab_size)


		self.wordToIx = {tok: i for i, tok in enumerate(sorted(vocab))}
	

	def forward(self, context_ids):
		embedded = self.embedding(context_ids)
		flattened = embedded.reshape(embedded.size(0), -1)
		hidden = torch.tanh(self.hidden(flattened))
		return self.output(hidden)

	def fit(self, sentences, epochs=300, lr=0.01):
		pairs = []

		for sentence in sentences:
			tokens = ["<s>"] * self.context_size + sentence + ["</s>"]
			ids = [self.wordToIx[t] for t in tokens]

			for i in range(self.context_size, len(ids)):
				pairs.append((ids[i-self.context_size:i], ids[i]))

		contexts = torch.tensor([p[0] for p in pairs])
		targets  = torch.tensor([p[1] for p in pairs])

		optimizer = torch.optim.Adam(self.parameters(), lr=lr)
		lossFn = nn.CrossEntropyLoss()

		super().train()

		for epoch in range(epochs):
			optimizer.zero_grad()
			logits = self.forward(contexts)
			loss = lossFn(logits, targets)
			loss.backward()
			optimizer.step()

			if (epoch + 1) % 100 == 0:
				print(f'  epoch {epoch+1:>4}  loss = {loss.item():.3f}')

	@torch.no_grad()
	def probability(self, token, t1, t2):
		context = torch.tensor([[self.wordToIx[t1], self.wordToIx[t2]]])

		logits = self.forward(context)
		probs = torch.softmax(logits, dim=-1)

		return probs[0, self.wordToIx[token]].item()

	@torch.no_grad()
	def perplexity(self, sentences):
		log_prob = 0
		total_tokens = 0

		super().eval()

		for sentence in sentences:
			tokens = ["<s>"] * self.context_size + sentence + ["</s>"]

			for i in range(self.context_size, len(tokens)):
				context = tokens[i-self.context_size: i]
				token = tokens[i]

				context_ids = torch.tensor([[self.wordToIx[t] for t in context]])

				logits = self.forward(context_ids)
				probs = torch.log_softmax(logits, dim=-1)

				log_prob += probs[0, self.wordToIx[token]].item()
				total_tokens += 1

		if total_tokens == 0:
			return float("inf")

		return math.exp(-log_prob / total_tokens)

In [8]:
ff_trigram_results = {}


for epoch in [50, 100, 200, 300, 400]:
	print(f"epochs = {epoch}")
	torch.manual_seed(0)

	ff_trigram = FF_Trigram(vocab)
	ff_trigram.fit(t_questions_filtered, epochs=epoch)
	fftg_ppl = ff_trigram.perplexity(val_unk)

	ff_trigram_results[epoch] = fftg_ppl

	print(f'Feedforward LM validation perplexity: {fftg_ppl:.2f}\n')


epochs = 50
Feedforward LM validation perplexity: 31.62

epochs = 100
  epoch  100  loss = 2.514
Feedforward LM validation perplexity: 21.97

epochs = 200
  epoch  100  loss = 2.514
  epoch  200  loss = 2.003
Feedforward LM validation perplexity: 21.31

epochs = 300
  epoch  100  loss = 2.514
  epoch  200  loss = 2.003
  epoch  300  loss = 1.777
Feedforward LM validation perplexity: 25.44

epochs = 400
  epoch  100  loss = 2.514
  epoch  200  loss = 2.003
  epoch  300  loss = 1.777
  epoch  400  loss = 1.654
Feedforward LM validation perplexity: 31.69



In [9]:
torch.manual_seed(0)

ff_trigram = FF_Trigram(vocab)
ff_trigram.fit(t_questions_filtered, epochs=200)

  epoch  100  loss = 2.514
  epoch  200  loss = 2.003


In [10]:
# Gather all previously calculated validation perplexities.
perplexity_rows = [
    {"Model": "Unigram", "Configuration": "Baseline", "Perplexity": unigram_ppl},
    {"Model": "Bigram", "Configuration": "Add-1 smoothing", "Perplexity": bigram_ppl},
]

for alpha, ppl in trigram_results.items():
    perplexity_rows.append({
        "Model": "Trigram", "Configuration": f"alpha={alpha}", "Perplexity": ppl
    })

for epochs, ppl in ff_trigram_results.items():
    perplexity_rows.append({
        "Model": "Feedforward trigram", "Configuration": f"epochs={epochs}", "Perplexity": ppl
    })

perplexity_table = pd.DataFrame(perplexity_rows)
display(perplexity_table.style.format({"Perplexity": "{:.2f}"}))

,Model,Configuration,Perplexity
0,Unigram,Baseline,185.07
1,Bigram,Add-1 smoothing,52.21
2,Trigram,alpha=1,103.84
3,Trigram,alpha=0.1,43.01
4,Trigram,alpha=0.01,25.76
5,Trigram,alpha=0.001,24.73
6,Feedforward trigram,epochs=50,31.62
7,Feedforward trigram,epochs=100,21.97
8,Feedforward trigram,epochs=200,21.31
9,Feedforward trigram,epochs=300,25.44


# High- and Low-perplexity questions


The different language models are split into different tables that holds the perplexities for each of the questions provided in <i>val_unk</i>. then the 5 higest and lowest perplexity questions for each language model are selected and stored.


In [11]:



question_results = df_val_ko[['question']].copy()




# Create separate DataFrames
unigram_results = df_val_ko[['question']].copy()
unigram_results["perplexity"] = [unigram.perplexity([q]) for q in val_unk]

bigram_results = df_val_ko[['question']].copy()
bigram_results["perplexity"] = [bigram.perplexity([q]) for q in val_unk]

trigram_results = df_val_ko[['question']].copy()
trigram_results["perplexity"] = [trigram.perplexity([q], 0.01) for q in val_unk]

ff_trigram_results = df_val_ko[['question']].copy()
ff_trigram_results["perplexity"] = [ff_trigram.perplexity([q]) for q in val_unk]


# Combine the 2 lowest and 2 highest for each model
def get_extremes(df, n):
    lowest = df.nsmallest(n, "perplexity").copy()
    lowest["Rank"] = "Lowest"

    highest = df.nlargest(n, "perplexity").copy()
    highest["Rank"] = "Highest"

    return pd.concat([lowest, highest], ignore_index=True)[
        ["Rank", "question", "perplexity"]
    ]

# Create final DataFrames
unigram_df = get_extremes(unigram_results, 5)
bigram_df = get_extremes(bigram_results, 5)
trigram_df = get_extremes(trigram_results, 5)
ff_trigram_df = get_extremes(ff_trigram_results, 5)





print("Unigram")
display(unigram_df)

print("Bigram")
display(bigram_df)

print("Trigram")
display(trigram_df)

print("FF Trigram")
display(ff_trigram_df)

Unigram


,Rank,question,perplexity
0,Lowest,세상에서 가장 넓은 산은 무엇인가?,57.857549
1,Lowest,2011년 이란의 지도자는 누구인가?,65.407591
2,Lowest,KKK의 지도자는 누구인가?,65.407591
3,Lowest,다이묘란 무엇인가?,69.315407
4,Lowest,2019년 기준 세상에서 가장 희귀한 병은 무엇인가?,71.610088
5,Highest,부커 T. 워싱턴은 총 몇번의 결혼을 했나요?,716.071976
6,Highest,부커 T. 워싱턴은 총 몇번의 결혼을 했나요?,716.071976
7,Highest,인도네시아에서는 동성결혼이 합법적인가?,673.184051
8,Highest,인도네시아에서는 동성결혼이 합법적인가?,673.184051
9,Highest,근대의 식민지는 어느 시대에 본격화 되었나요?,572.951138


Bigram


,Rank,question,perplexity
0,Lowest,세상에서 가장 큰 대학교는 무엇인가?,8.070707
1,Lowest,2011년 이란의 지도자는 누구인가?,9.496725
2,Lowest,KKK의 지도자는 누구인가?,9.496725
3,Lowest,이탈리아에서 가장 높은 산은 무엇인가?,10.513804
4,Lowest,세상에서 가장 넓은 산은 무엇인가?,10.657639
5,Highest,"임시정부는 종전의 정부가 무너진 후, 무정부 상태를 해소하기 위해 임시로 구성된 정...",298.249706
6,Highest,인체가 자외선에 많이 노출된다면 어떤 변화가 일어나는가?,291.955630
7,Highest,인체가 자외선에 많이 노출된다면 어떤 변화가 일어나는가?,291.955630
8,Highest,불고기는 소의 어느부위를 사용하는가?,271.254509
9,Highest,핵융합은 상용화 가능성이 있는가?,264.558662


Trigram


,Rank,question,perplexity
0,Lowest,중국에서 가장 인구밀도가 높은 도시는 어디인가?,2.561292
1,Lowest,세상에서 가장 큰 대학교는 무엇인가?,2.840971
2,Lowest,이탈리아에서 가장 높은 산은 무엇인가?,2.849162
3,Lowest,2011년 이란의 지도자는 누구인가?,3.009340
4,Lowest,KKK의 지도자는 누구인가?,3.009340
5,Highest,불고기는 소의 어느부위를 사용하는가?,515.133828
6,Highest,냉전 기간 동안 미국을 돕는 나라는?,441.493126
7,Highest,인체가 자외선에 많이 노출된다면 어떤 변화가 일어나는가?,433.949208
8,Highest,인체가 자외선에 많이 노출된다면 어떤 변화가 일어나는가?,433.949208
9,Highest,지자기 폭풍이 지구에 일어났을때 지구에선 어떤 현상이 일어나는가?,430.577658


FF Trigram


,Rank,question,perplexity
0,Lowest,중국에서 가장 인구밀도가 높은 도시는 어디인가?,2.177821
1,Lowest,이탈리아에서 가장 높은 산은 무엇인가?,2.425973
2,Lowest,세상에서 가장 큰 대학교는 무엇인가?,2.505791
3,Lowest,2011년 이란의 지도자는 누구인가?,2.516465
4,Lowest,KKK의 지도자는 누구인가?,2.516465
5,Highest,불고기는 소의 어느부위를 사용하는가?,1263.417973
6,Highest,핵융합은 상용화 가능성이 있는가?,1166.361758
7,Highest,인체가 자외선에 많이 노출된다면 어떤 변화가 일어나는가?,982.811011
8,Highest,인체가 자외선에 많이 노출된다면 어떤 변화가 일어나는가?,982.811011
9,Highest,신화에는 그리스 신들이 몇 명이나 있을까?,561.577578


## Analysis of question perplexity

We analyzed why some questions may or may not lead to high/low perplexity. Our assumption was that for each of the models, if the n-grams of the question, appears more often in the training sets, the would lead to lower perplexity, and opposite, n-gram appears less often would lead to high perplexity.


This is being counted how many times <i>(combined for each n-gram in question)</i> the n-grams appear in the training set, and is put into a table for presentation.

Here it can be observed, that for all <i>(all of our chosen examples)</i>, the hypothesis holds.



In [ ]:
def add_question_tokenized_to_df(df):
    df["tokenized"] = [tokenizer.tokenize(df["question"].iloc[i]) for i in range(len(df))]

add_question_tokenized_to_df(unigram_df)
add_question_tokenized_to_df(bigram_df)
add_question_tokenized_to_df(trigram_df)
add_question_tokenized_to_df(ff_trigram_df)

def n_gram(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n)] # ignores the last ? token at the end


unisums = []
bisums = []
trisums = []
ff_trisums = []

for i in range(len(unigram_df)):
	uni_sum = 0
	for token in unigram_df["tokenized"].iloc[i]:
		uni_sum += unigram.count[token]

	unisums.append(uni_sum)

for i in range(len(bigram_df)):
	bi_sum = 0
	bi_tokens = n_gram(bigram_df["tokenized"].iloc[i], 2)
	for token in bi_tokens:
		bi_sum += bigram.bigram_count[token]

	bisums.append(bi_sum)

for i in range(len(trigram_df)):
	tri_sum = 0
	tri_tokens = n_gram(trigram_df["tokenized"].iloc[i], 3)
	for token in tri_tokens:
		tri_sum += trigram.trigram_count[token]

	trisums.append(tri_sum)

for i in range(len(ff_trigram_df)):
	ff_tri_sum = 0
	tri_tokens = n_gram(ff_trigram_df["tokenized"].iloc[i], 3)
	for token in tri_tokens:
		ff_tri_sum += trigram.trigram_count[token]

	ff_trisums.append(ff_tri_sum)
    


unigram_df["ngram_appearances"] = unisums
bigram_df["ngram_appearances"] = bisums
trigram_df["ngram_appearances"] = trisums
ff_trigram_df["ngram_appearances"] = ff_trisums



display(unigram_df)
display(bigram_df)
display(trigram_df)
display(ff_trigram_df)

,Rank,question,perplexity,tokenized,ngram_appearances
0,Lowest,세상에서 가장 넓은 산은 무엇인가?,57.857549,"[세, ##상, ##에서, 가장, 넓, ##은, 산, ##은, 무, ##엇, ##인...",10711
1,Lowest,2011년 이란의 지도자는 누구인가?,65.407591,"[2011년, 이란, ##의, 지, ##도, ##자는, 누, ##구, ##인, ##...",7553
2,Lowest,KKK의 지도자는 누구인가?,65.407591,"[KK, ##K, ##의, 지, ##도, ##자는, 누, ##구, ##인, ##가, ?]",7553
3,Lowest,다이묘란 무엇인가?,69.315407,"[다, ##이, ##묘, ##란, 무, ##엇, ##인, ##가, ?]",7737
4,Lowest,2019년 기준 세상에서 가장 희귀한 병은 무엇인가?,71.610088,"[2019, ##년, 기준, 세, ##상, ##에서, 가장, 희, ##귀, ##한,...",10153
5,Highest,부커 T. 워싱턴은 총 몇번의 결혼을 했나요?,716.071976,"[부, ##커, T, ., 워, ##싱, ##턴, ##은, 총, 몇, ##번, ##...",5630
6,Highest,부커 T. 워싱턴은 총 몇번의 결혼을 했나요?,716.071976,"[부, ##커, T, ., 워, ##싱, ##턴, ##은, 총, 몇, ##번, ##...",5630
7,Highest,인도네시아에서는 동성결혼이 합법적인가?,673.184051,"[인, ##도, ##네, ##시아, ##에서는, 동, ##성, ##결, ##혼, #...",5308
8,Highest,인도네시아에서는 동성결혼이 합법적인가?,673.184051,"[인, ##도, ##네, ##시아, ##에서는, 동, ##성, ##결, ##혼, #...",5308
9,Highest,근대의 식민지는 어느 시대에 본격화 되었나요?,572.951138,"[근, ##대의, 식, ##민, ##지는, 어느, 시대, ##에, 본, ##격, #...",4326


,Rank,question,perplexity,tokenized,ngram_appearances
0,Lowest,세상에서 가장 큰 대학교는 무엇인가?,8.070707,"[세, ##상, ##에서, 가장, 큰, 대학교, ##는, 무, ##엇, ##인, #...",3628
1,Lowest,2011년 이란의 지도자는 누구인가?,9.496725,"[2011년, 이란, ##의, 지, ##도, ##자는, 누, ##구, ##인, ##...",2201
2,Lowest,KKK의 지도자는 누구인가?,9.496725,"[KK, ##K, ##의, 지, ##도, ##자는, 누, ##구, ##인, ##가, ?]",2201
3,Lowest,이탈리아에서 가장 높은 산은 무엇인가?,10.513804,"[이탈리아, ##에서, 가장, 높은, 산, ##은, 무, ##엇, ##인, ##가, ?]",3349
4,Lowest,세상에서 가장 넓은 산은 무엇인가?,10.657639,"[세, ##상, ##에서, 가장, 넓, ##은, 산, ##은, 무, ##엇, ##인...",3566
5,Highest,"임시정부는 종전의 정부가 무너진 후, 무정부 상태를 해소하기 위해 임시로 구성된 정...",298.249706,"[임, ##시, ##정, ##부는, 종, ##전, ##의, 정, ##부가, 무, #...",368
6,Highest,인체가 자외선에 많이 노출된다면 어떤 변화가 일어나는가?,291.955630,"[인, ##체가, 자, ##외, ##선, ##에, 많이, 노, ##출, ##된다, ...",14
7,Highest,인체가 자외선에 많이 노출된다면 어떤 변화가 일어나는가?,291.955630,"[인, ##체가, 자, ##외, ##선, ##에, 많이, 노, ##출, ##된다, ...",14
8,Highest,불고기는 소의 어느부위를 사용하는가?,271.254509,"[불, ##고, ##기는, 소, ##의, 어느, ##부, ##위를, 사용하는, ##...",4
9,Highest,핵융합은 상용화 가능성이 있는가?,264.558662,"[핵, ##융, ##합, ##은, 상, ##용, ##화, 가, ##능, ##성이, ...",93


,Rank,question,perplexity,tokenized,ngram_appearances
0,Lowest,중국에서 가장 인구밀도가 높은 도시는 어디인가?,2.561292,"[중국, ##에서, 가장, 인구, ##밀, ##도가, 높은, 도시, ##는, 어, ...",801
1,Lowest,세상에서 가장 큰 대학교는 무엇인가?,2.840971,"[세, ##상, ##에서, 가장, 큰, 대학교, ##는, 무, ##엇, ##인, #...",1775
2,Lowest,이탈리아에서 가장 높은 산은 무엇인가?,2.849162,"[이탈리아, ##에서, 가장, 높은, 산, ##은, 무, ##엇, ##인, ##가, ?]",1538
3,Lowest,2011년 이란의 지도자는 누구인가?,3.009340,"[2011년, 이란, ##의, 지, ##도, ##자는, 누, ##구, ##인, ##...",737
4,Lowest,KKK의 지도자는 누구인가?,3.009340,"[KK, ##K, ##의, 지, ##도, ##자는, 누, ##구, ##인, ##가, ?]",737
5,Highest,불고기는 소의 어느부위를 사용하는가?,515.133828,"[불, ##고, ##기는, 소, ##의, 어느, ##부, ##위를, 사용하는, ##...",0
6,Highest,냉전 기간 동안 미국을 돕는 나라는?,441.493126,"[냉, ##전, 기, ##간, 동안, 미국, ##을, 돕, ##는, 나, ##라는, ?]",1
7,Highest,인체가 자외선에 많이 노출된다면 어떤 변화가 일어나는가?,433.949208,"[인, ##체가, 자, ##외, ##선, ##에, 많이, 노, ##출, ##된다, ...",0
8,Highest,인체가 자외선에 많이 노출된다면 어떤 변화가 일어나는가?,433.949208,"[인, ##체가, 자, ##외, ##선, ##에, 많이, 노, ##출, ##된다, ...",0
9,Highest,지자기 폭풍이 지구에 일어났을때 지구에선 어떤 현상이 일어나는가?,430.577658,"[지, ##자, ##기, 폭, ##풍, ##이, 지, ##구, ##에, 일, ##어...",13


,Rank,question,perplexity,tokenized,ngram_appearances
0,Lowest,중국에서 가장 인구밀도가 높은 도시는 어디인가?,2.177821,"[중국, ##에서, 가장, 인구, ##밀, ##도가, 높은, 도시, ##는, 어, ...",801
1,Lowest,이탈리아에서 가장 높은 산은 무엇인가?,2.425973,"[이탈리아, ##에서, 가장, 높은, 산, ##은, 무, ##엇, ##인, ##가, ?]",1538
2,Lowest,세상에서 가장 큰 대학교는 무엇인가?,2.505791,"[세, ##상, ##에서, 가장, 큰, 대학교, ##는, 무, ##엇, ##인, #...",1775
3,Lowest,2011년 이란의 지도자는 누구인가?,2.516465,"[2011년, 이란, ##의, 지, ##도, ##자는, 누, ##구, ##인, ##...",737
4,Lowest,KKK의 지도자는 누구인가?,2.516465,"[KK, ##K, ##의, 지, ##도, ##자는, 누, ##구, ##인, ##가, ?]",737
5,Highest,불고기는 소의 어느부위를 사용하는가?,1263.417973,"[불, ##고, ##기는, 소, ##의, 어느, ##부, ##위를, 사용하는, ##...",0
6,Highest,핵융합은 상용화 가능성이 있는가?,1166.361758,"[핵, ##융, ##합, ##은, 상, ##용, ##화, 가, ##능, ##성이, ...",1
7,Highest,인체가 자외선에 많이 노출된다면 어떤 변화가 일어나는가?,982.811011,"[인, ##체가, 자, ##외, ##선, ##에, 많이, 노, ##출, ##된다, ...",0
8,Highest,인체가 자외선에 많이 노출된다면 어떤 변화가 일어나는가?,982.811011,"[인, ##체가, 자, ##외, ##선, ##에, 많이, 노, ##출, ##된다, ...",0
9,Highest,신화에는 그리스 신들이 몇 명이나 있을까?,561.577578,"[신, ##화, ##에는, 그리스, 신, ##들이, 몇, 명, ##이나, 있을, #...",1
